# 05_데이터_통합

**목적**: 레포 정본 `data/final/fatigue_with_index.csv` (77 cols, utf-8-sig) 에 로컬 `legacy_최종.csv` 에만 존재하는 컬럼 3개를 흡수.

**머지할 컬럼**: `혹사정도` (float), `회복실패여부` (0/1), `휴식_그룹` ('회복실패'/'회복충분')

**조인 키**: `(선수, 날짜)` — 사전 검증에서 양쪽 17,528행 1:1 매칭 확인됨.

**출력**: `data/final/fatigue_with_index.csv` 를 80컬럼 버전으로 덮어쓰기 (utf-8-sig).

**Idempotent**: 이미 3컬럼이 존재하면 머지 건너뜀.

In [ ]:
import pandas as pd
from pathlib import Path

FINAL = Path('../data/final')
CANONICAL = FINAL / 'fatigue_with_index.csv'
LEGACY    = FINAL / 'legacy_최종.csv'

NEW_COLS = ['혹사정도', '회복실패여부', '휴식_그룹']
JOIN_KEYS = ['선수', '날짜']

In [ ]:
df_canon  = pd.read_csv(CANONICAL)
df_legacy = pd.read_csv(LEGACY, encoding='utf-8-sig')

print(f'canonical: {df_canon.shape}')
print(f'legacy   : {df_legacy.shape}')

In [ ]:
# 조인 키 1:1 검증
assert df_canon[JOIN_KEYS].duplicated().sum() == 0,  'canonical에 (선수, 날짜) 중복'
assert df_legacy[JOIN_KEYS].duplicated().sum() == 0, 'legacy에 (선수, 날짜) 중복'

probe = df_canon[JOIN_KEYS].merge(df_legacy[JOIN_KEYS], on=JOIN_KEYS, how='outer', indicator=True)
print(probe['_merge'].value_counts())
assert (probe['_merge'] == 'both').all(), '양쪽 조인 키 불일치 발견'
print('1:1 join key match: OK')

In [ ]:
# Idempotency: 이미 머지된 상태면 skip
missing = [c for c in NEW_COLS if c not in df_canon.columns]
if not missing:
    print('이미 모든 컬럼 존재 — skip')
    df_merged = df_canon
else:
    print(f'추가 대상: {missing}')
    df_merged = df_canon.merge(
        df_legacy[JOIN_KEYS + missing],
        on=JOIN_KEYS, how='left', validate='one_to_one',
    )
    print(f'머지 후 shape: {df_merged.shape}')

In [ ]:
# 결과 검증
assert len(df_merged) == len(df_canon), '행 수 변경 — left join 무결성 깨짐'
for c in NEW_COLS:
    assert c in df_merged.columns, f'{c} 컬럼 누락'
    nn = df_merged[c].isna().sum()
    print(f'  {c}: NaN={nn}, sample={df_merged[c].dropna().unique()[:5]}')

print(f'최종 컬럼 수: {df_merged.shape[1]} (예상 80)')

In [ ]:
# 덮어쓰기 (utf-8-sig)
df_merged.to_csv(CANONICAL, index=False, encoding='utf-8-sig')
print(f'saved: {CANONICAL.resolve()}')

# 재읽기 검증
df_check = pd.read_csv(CANONICAL)
print(f'재읽기 shape: {df_check.shape}')
assert df_check.shape == df_merged.shape

## 실행 결과 요약

- `fatigue_with_index.csv`: 77 → 80 columns
- 추가된 3컬럼은 `(선수, 날짜)` 기준 1:1로 안전하게 매핑됨 (NaN 0)
- 모든 기존 컬럼/행은 보존

## 후속 영향

- `app/역전점_앱.py`, `notebooks/02_역전점_구하기_2차_모델링.ipynb`, `notebooks/04_시각화.ipynb` 등 정본 CSV를 읽는 코드는 변경 없이 호환 (컬럼 추가는 비파괴적).
- 신규 3컬럼을 활용하는 분석은 별도 셀/노트북에서 추가.